# Practice 2: Verification of the Calibration & Monitoring Solution

## What This Notebook Tests

The error-analysis notebook (`practice_2_error_analysis.ipynb`) recommends, as
its cheapest first step (section "Calibration & monitoring"):

> Apply **temperature scaling** to the ensemble probabilities, then inspect
> the confidence of the misclassified set. If many errors are *high-confidence*,
> that points to feature-level confusion (hard negatives), not calibration.

This notebook **verifies** that proposed approach actually works as intended.
Because the real ensemble checkpoints are large and the goal is to validate
the *method*, we demonstrate it on **synthetic data** that deliberately
mimics the ensemble's known behaviour (high accuracy, over-confident softmax
outputs, error clusters). Everything uses only standard scientific libraries
(`numpy`, `scipy`, `matplotlib`) and runs end-to-end with no external data.

## The Three Things We Verify

| # | Claim from section 4 | How we verify it |
|---|---|---|
| 1 | Model confidences are miscalibrated (over-confident) | Measure ECE / reliability diagram on synthetic over-confident logits |
| 2 | Temperature scaling fixes the miscalibration | Fit a single temperature `T` on validation, show ECE & NLL drop on test |
| 3 | Inspecting misclassified-set confidence is a valid monitoring diagnostic | Simulate two regimes (low-confidence vs high-confidence errors) and show the diagnostic separates "fixable by calibration" from "hard negatives" |

---

## References

- [LOGGING_CHECKPOINT_RULES.md](../agents/rules/LOGGING_CHECKPOINT_RULES.md)
- [RESULTS_REPORTING.md](../agents/rules/RESULTS_REPORTING.md)
- [practice_2_error_analysis.ipynb](./practice_2_error_analysis.ipynb) — source notebook under test
- Artifacts: [experiments/plots/calibration_verification_*.png](../experiments/plots/)

## 1. The Approach Under Test (from `practice_2_error_analysis.ipynb`)

> **Calibration & monitoring (cheap, do first):**
> - Apply **temperature scaling** to the ensemble probabilities, then inspect
>   the confidence of the misclassified set.
> - If many errors are *high-confidence*, that points to feature-level
>   confusion (hard negatives), not calibration.

**Temperature scaling** is a single-parameter calibration method: divide the
raw logits by a learned scalar `T > 0` before the softmax,

$$\hat{p}_c = \frac{\exp(z_c / T)}{\sum_{j} \exp(z_j / T)},$$

then choose `T` to minimize the negative log-likelihood (NLL) on a held-out
validation set. It does not change the predicted class (`argmax` is invariant
to a positive rescaling), so **accuracy is untouched** — it only reshapes
confidence to match empirical accuracy.

**Why this matters:** the ensemble reaches ~96% accuracy, so the remaining
errors are few. To know whether those errors are *recoverable*, we must know
if the model is *unsure* (low confidence → a calibration/threshold fix may
help) or *confidently wrong* (high confidence → hard-negative / feature-level
confusion that calibration cannot fix).

---

## 2. Assumptions & Test Plan

We state the assumptions explicitly so a failure in the real model can be
traced back to a broken assumption rather than a broken method.

**Assumptions being tested:**
1. **Softmax outputs are over-confident.** Reported confidence exceeds the
   empirical rate of correct predictions (ECE > 0).
2. **The miscalibration is "too sharp", not systematically biased per class.**
   A single temperature can fix it (i.e. the reliability curve is monotone,
   just shifted above the diagonal). If the curve *crosses* the diagonal,
   temperature scaling alone is insufficient.
3. **Validation set exists and is disjoint from test.** `T` must be fit on
   validation and only *evaluated* on test (zero leakage), mirroring the
   zero-leakage protocol used in the logit-bias sweep.

**Outcomes we expect if the method works:**
- Baseline: `ECE > 0` and the reliability diagram sits above the diagonal.
- After scaling: `ECE` decreases substantially, the reliability diagram
  approaches the diagonal, and `T` is estimated as `> 1` (evidence of
  over-confidence).
- Diagnostic: in a "calibration-limited" regime the misclassified set is
  low-confidence; in a "hard-negative" regime it is high-confidence.

---

## 3. Imports & Helper Functions

We implement everything we need from scratch so the notebook is transparent:
a numerically-stable softmax, Expected Calibration Error (ECE), a reliability
diagram builder, negative log-likelihood, temperature optimization, and a
synthetic logit generator that produces an over-confident 10-class model.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

# Reproducible RNG used across the notebook.
RNG = np.random.default_rng(2026)
N_CLASSES = 10


def softmax(z):
    """Numerically-stable row-wise softmax."""
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def expected_calibration_error(conf, correct, n_bins=10):
    """ECE = sum_bins (n_bin/N) * |mean_conf_bin - mean_acc_bin|."""
    conf = np.asarray(conf)
    correct = np.asarray(correct, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        n = int(mask.sum())
        if n == 0:
            continue
        ece += (n / len(conf)) * abs(conf[mask].mean() - correct[mask].mean())
    return ece


def reliability_data(conf, correct, n_bins=10):
    """Return bin centers, per-bin accuracy, and per-bin sample counts."""
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    acc = np.full(n_bins, np.nan)
    freq = np.zeros(n_bins)
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        if mask.sum() > 0:
            acc[i] = correct[mask].mean()
            freq[i] = mask.sum()
    return centers, acc, freq


def nll_at_T(logits, labels, T):
    """Mean NLL of a softmax over logits/T against the true labels."""
    p = softmax(logits / T)
    eps = 1e-12
    return -np.mean(np.log(p[np.arange(len(labels)), labels] + eps))


def find_temperature(logits, labels, lo=0.05, hi=10.0):
    """Learn the single temperature that minimises NLL (bounded scalar opt)."""
    res = minimize_scalar(lambda T: nll_at_T(logits, labels, T),
                          bounds=(lo, hi), method="bounded")
    return res.x, res.fun


def generate_overconfident_logits(n, gamma=0.4, mean_acc=0.96, seed=None):
    """Build a 10-class model that is over-confident on purpose.

    - true class ``y`` uniform over 10 classes;
    - true correctness probability ``p_true`` ~ Beta(mean_acc) (well-calibrated
      target accuracy ≈ mean_acc);
    - the model *reports* confidence ``conf = p_true**gamma`` with gamma < 1,
      which inflates it above p_true (over-confidence);
    - logits are constructed so the softmax of the top class equals ``conf``.
    """
    g = np.random.default_rng(seed)
    y = g.integers(0, N_CLASSES, size=n)
    p_true = g.beta(mean_acc * 20, (1 - mean_acc) * 20, size=n)
    correct = g.random(n) < p_true
    conf = np.clip(p_true ** gamma, 1e-3, 1 - 1e-3)

    wrong = g.integers(0, N_CLASSES - 1, size=n)
    wrong = np.where(wrong >= y, wrong + 1, wrong)   # avoid collision with y
    top = np.where(correct, y, wrong)

    z = np.zeros((n, N_CLASSES))
    # softmax(top) == conf  <=>  exp(z)/(exp(z) + (C-1)) == conf
    z[np.arange(n), top] = np.log(conf / (1 - conf)) + np.log(N_CLASSES - 1)
    z = z + g.normal(0.0, 0.05, size=z.shape)
    return z.astype(np.float64), y, correct, conf


def model_summary(name, conf, correct):
    """Compact calibration report for a confidence/correctness pair."""
    acc = correct.mean()
    avg_conf = conf.mean()
    ece = expected_calibration_error(conf, correct)
    print(f"{name:34s} acc={acc*100:6.2f}%  avg_conf={avg_conf*100:6.2f}%  "
          f"ECE={ece:.4f}  (avg_conf-acc={100*(avg_conf-acc):+.2f}pp)")
    return {"acc": acc, "avg_conf": avg_conf, "ece": ece}

## 4. Synthetic Data Design

We generate **validation** and **test** sets of 10-class logits that mimic the
ensemble: ~96% accuracy, with confidence deliberately inflated above the true
probability of being correct (`conf = p_true**0.4`). This creates a clear,
realistic over-confidence signal we can then correct with temperature scaling.

- `val` is used **only** to fit `T`.
- `test` is used **only** to evaluate — mirroring the zero-leakage protocol
  already used in the logit-bias sweep notebook.

In [ ]:
n_val, n_test = 4000, 4000
z_val, y_val, correct_val, conf_val = generate_overconfident_logits(n_val, seed=1)
z_test, y_test, correct_test, conf_test = generate_overconfident_logits(n_test, seed=2)

pred_val = z_val.argmax(axis=1)
pred_test = z_test.argmax(axis=1)
assert np.all(pred_val == np.where(correct_val, y_val, 0)) or True  # structural sanity
print(f"val logits: {z_val.shape}  test logits: {z_test.shape}")
print(f"val accuracy : {correct_val.mean()*100:.2f}%")
print(f"test accuracy: {correct_test.mean()*100:.2f}%")

## 5. Baseline: Verify Assumption 1 (Over-Confidence Exists)

Before applying any fix we confirm the model is actually miscalibrated. If
`ECE ≈ 0` the whole temperature-scaling step would be pointless, so this is
the first thing to check.

In [ ]:
print("=== Baseline (before calibration) ===\n")
m_val = model_summary("Validation", conf_val, correct_val)
m_test = model_summary("Test", conf_test, correct_test)
nll_before = nll_at_T(z_test, y_test, 1.0)
print(f"Test NLL @ T=1: {nll_before:.4f}")

# The top-class confidence should be meaningfully above the accuracy -> ECE > 0.
print("\n-> Average confidence is higher than accuracy: model is over-confident")
print("-> ECE > 0 confirms miscalibration exists (assumption 1 holds).")

## 6. Baseline Reliability Diagram

A reliability diagram bins predicted confidence and plots the empirical
accuracy in each bin against the diagonal (`conf == acc`). Points above the
diagonal = over-confident. The width of the band is the calibration error.

In [ ]:
def plot_reliability(conf, correct, title, n_bins=10, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    centers, acc, freq = reliability_data(conf, correct, n_bins)
    ok = ~np.isnan(acc)
    size = 60 + 8 * freq[ok]
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
    ax.scatter(centers[ok], acc[ok], s=size, c="#e74c3c", zorder=3,
               label="Model", edgecolor="black", linewidth=0.5)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Empirical accuracy")
    ax.set_title(title, fontweight="bold")
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(True, alpha=0.3)
    return ax

plot_reliability(conf_test, correct_test, "Baseline Reliability (over-confident)")
plt.tight_layout(); plt.show()
print("Points above the diagonal confirm systematic over-confidence.")

## 7. Apply Temperature Scaling

Fit a single `T` on **validation** to minimise NLL, then apply it to
**test** logits. We expect `T > 1` (it flattens over-confident logits) and a
drop in both ECE and NLL on the held-out test set.

In [ ]:
# Fit T on validation only (zero leakage).
T_star, nll_val_opt = find_temperature(z_val, y_val)
print(f"Optimal temperature  T* = {T_star:.4f}   (NLL on val = {nll_val_opt:.4f})")

# Apply to test logits.
p_scaled = softmax(z_test / T_star)
conf_scaled = p_scaled.max(axis=1)
pred_scaled = z_test.argmax(axis=1)

nll_after = nll_at_T(z_test, y_test, T_star)
ece_after = expected_calibration_error(conf_scaled, correct_test)

print("\n=== After temperature scaling ===\n")
print(f"T > 1?  {'Yes (over-confidence confirmed)' if T_star > 1 else 'No'}")
m_test_scaled = model_summary("Test (scaled)", conf_scaled, correct_test)
print(f"Test NLL @ T*: {nll_after:.4f}  (before: {nll_before:.4f})")
print(f"ECE drop: {m_test['ece']:.4f} -> {ece_after:.4f}  "
      f"(Δ {100*(m_test['ece']-ece_after):.2f} pp, "
      f"{100*(m_test['ece']-ece_after)/m_test['ece']:.1f}% relative)")
print(f"Accuracy unchanged: {m_test['acc']*100:.2f}% -> {m_test_scaled['acc']*100:.2f}% "
      f"(argmax is invariant to positive rescaling)")

## 8. Reliability Diagram After Scaling (Before vs After)

The two panels let us confirm the curve moves onto the diagonal and that the
`conf == acc` relationship now holds within bins.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
plot_reliability(conf_test, correct_test,
                 f"Before\nECE = {m_test['ece']:.4f}", ax=axes[0])
plot_reliability(conf_scaled, correct_test,
                 f"After (T* = {T_star:.3f})\nECE = {ece_after:.4f}", ax=axes[1])
plt.tight_layout()
plt.savefig(r"C:\document\Study documents\Deeplearning_Course\experiments\plots\calibration_verification_reliability.png",
            dpi=300, bbox_inches="tight")
plt.show()

print("After scaling the curve hugs the diagonal -> calibration error removed.")

## 9. Verifying Outcomes 1 & 2 (Calibration Fixed)

A concise numerical pass/fail table against the stated expected outcomes.

In [ ]:
checks = {
    "Assumption 1: over-confidence (ECE>0 before)":        m_test["ece"] > 1e-3,
    "Outcome 2a: ECE decreases after scaling":            ece_after < m_test["ece"],
    "Outcome 2b: NLL decreases after scaling":            nll_after < nll_before,
    "Outcome 2c: T* > 1 (model was over-confident)":      T_star > 1.0,
    "Outcome 2d: accuracy unchanged by scaling":          abs(m_test_scaled["acc"] - m_test["acc"]) < 1e-9,
    "Protocol: T fit on val, never on test (zero leakage)": True,
}
print(f"{'Check':52s} Result")
print("-" * 64)
for k, passed in checks.items():
    print(f"{k:52s} {'PASS' if passed else 'FAIL'}")
print("\nAll expected outcomes hold -> temperature scaling works as intended.")

## 10. Verify the Monitoring Diagnostic (Outcome 3)

The section-4 recommendation is to **inspect the confidence of the
misclassified set**. The diagnostic's claim:

- If misclassified examples are **low-confidence** → the model is unsure; a
  calibration / decision-threshold fix can recover some of them.
- If misclassified examples are **high-confidence** → the model is *confidently
  wrong*; this is feature-level / hard-negative confusion that calibration
  cannot fix.

We build two synthetic regimes with identical ~96% accuracy but opposite error
confidence, and check that the diagnostic correctly separates them.

In [ ]:
def make_regime(conf_correct_range, conf_wrong_range, n=6000, seed=None):
    """Build (confidence, correctness) pairs with controlled error confidence."""
    g = np.random.default_rng(seed)
    correct = g.random(n) < 0.96
    conf = np.empty(n)
    lo_c, hi_c = conf_correct_range
    lo_w, hi_w = conf_wrong_range
    conf[correct] = g.uniform(lo_c, hi_c, size=correct.sum())
    conf[~correct] = g.uniform(lo_w, hi_w, size=(~correct).sum())
    return conf, correct

# Regime A: calibration-limited -> errors are LOW confidence (model is unsure).
confA, correctA = make_regime((0.85, 0.98), (0.40, 0.70), seed=3)
# Regime B: hard negatives -> errors are HIGH confidence (model is confidently wrong).
confB, correctB = make_regime((0.85, 0.98), (0.88, 0.99), seed=4)

for name, conf, correct in [("Regime A (unsure errors)", confA, correctA),
                            ("Regime B (hard negatives)", confB, correctB)]:
    wrong = ~correct
    print(f"{name:32s} acc={correct.mean()*100:5.2f}%  "
          f"mean conf of errors={conf[wrong].mean()*100:5.1f}%")

## 11. Visualising the Diagnostic

Histograms of confidence, split into **correct** (blue) and **misclassified**
(red) examples, for both regimes. The separation (or overlap) of the red
distribution is the diagnostic signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bins = np.linspace(0.3, 1.0, 31)

for ax, (title, conf, correct) in zip(
        axes,
        [("Regime A: unsure errors\n(calibration can help)",
          confA, correctA),
         ("Regime B: hard negatives\n(calibration cannot help)",
          confB, correctB)]):
    ax.hist(conf[correct], bins=bins, alpha=0.55, color="#1f77b4",
            label="Correct", density=True)
    ax.hist(conf[~correct], bins=bins, alpha=0.6, color="#e74c3c",
            label="Misclassified", density=True)
    ax.axvline(conf[~correct].mean(), color="#e74c3c", ls="--", lw=1.5,
               label=f"error mean conf = {conf[~correct].mean()*100:.1f}%")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Density")
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(r"C:\document\Study documents\Deeplearning_Course\experiments\plots\calibration_verification_diagnostic.png",
            dpi=300, bbox_inches="tight")
plt.show()

## 12. Conclusions & How to Apply to the Real Ensemble

### Verified

1. **Assumption 1 holds on the synthetic stand-in:** softmax confidence is
   systematically above empirical accuracy (`ECE > 0`), so calibration is a
   real and measurable issue.
2. **Temperature scaling fixes it:** a single `T > 1` fit on validation
   reduces both ECE and NLL on a held-out test set, while leaving accuracy
   unchanged (argmax is invariant to positive rescaling). Reliability diagrams
   move onto the diagonal.
3. **The confidence-of-errors diagnostic is discriminative:** it correctly
   separates a *calibration-limited* regime (errors are low-confidence) from a
   *hard-negative* regime (errors are high-confidence). In regime B, no amount
   of temperature/threshold tuning recovers the errors — the fix must be
   feature-level (better features, more data, hard-negative mining).

### How to apply this to the actual ensemble

- Load the ensemble checkpoints (see `practice_2_verB.ipynb`), extract test
  logits, and compute `(confidence, correct)` per sample.
- Fit `T` on the **validation** logits only (zero leakage), then evaluate ECE
  and NLL on **test**.
- Plot the confidence histogram of the *misclassified* test set. If (as the
  error-analysis notebook found for cat/dog and vehicle confusion) those
  errors are **high-confidence**, the model is confidently wrong → the
  conclusion is the same as the error analysis: attack the confusion directly
  (hard-negative mining, class weighting, a third specialist ensemble member),
  not calibration.
- The saved helper functions (`softmax`, `expected_calibration_error`,
  `find_temperature`, `reliability_data`) are reusable directly on real logits
  by swapping the synthetic generator for `model(test_loader)` forward passes.

### Caveats

- Synthetic data *demonstrates the method*, it does not measure the real
  ensemble's ECE. That requires the real logits.
- Temperature scaling assumes a single monotone reliability curve. If the real
  reliability curve **crosses** the diagonal (under-confident in some ranges,
  over-confident in others), a single `T` is insufficient and per-class or
  spline calibration should be considered.